<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/5obliczenia_podstawowe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_obliczenia_podstawowe.py

from __future__ import annotations

import json
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from statistics import mean, pstdev
from typing import Any, Callable, Final, NamedTuple

from modul_model_danych import Notowanie, Spolka, HistoriaCen


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


class ZakresCen(NamedTuple):
    minimum: float
    maksimum: float
    srednia: float


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class WynikObliczen:
    ticker: str

    srednia_kroczaca: float
    stopa_zwrotu: float
    sredni_wolumen: float
    zmiennosc: float

    liczba_notowan: int

    zakres_cen: ZakresCen

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )


class SredniaKroczaca:
    def __init__(
        self,
        okres: int = 20
    ) -> None:

        if okres <= 0:
            raise ValueError(
                "okres sredniej musi byc wiekszy od 0"
            )

        self.okres = okres

    def __call__(
        self,
        historia: HistoriaCen
    ) -> float:

        if not historia.notowania:
            raise ValueError(
                "brak notowan do obliczenia sredniej"
            )

        liczba: int = min(
            self.okres,
            len(historia.notowania)
        )

        ceny: list[float] = [
            n.close
            for n in historia.notowania[-liczba:]
        ]

        return float(
            mean(ceny)
        )


class ProstaStopaZwrotu:
    def __call__(
        self,
        historia: HistoriaCen
    ) -> float:

        if len(historia.notowania) < 2:
            return 0.0

        pierwsza: float = (
            historia.notowania[0].close
        )

        ostatnia: float = (
            historia.notowania[-1].close
        )

        if pierwsza == 0:
            raise ValueError(
                "pierwsza cena nie moze wynosic 0"
            )

        return (
            (ostatnia - pierwsza)
            / pierwsza
        ) * 100.0


class SredniWolumen:
    def __init__(
        self,
        okres: int | None = None
    ) -> None:

        if okres is not None and okres <= 0:
            raise ValueError(
                "okres musi byc wiekszy od 0"
            )

        self.okres = okres

    def __call__(
        self,
        historia: HistoriaCen
    ) -> float:

        notowania: list[Notowanie]

        if self.okres is None:
            notowania = historia.notowania
        else:
            notowania = historia.notowania[
                -self.okres:
            ]

        if not notowania:
            raise ValueError(
                "brak notowan do obliczenia wolumenu"
            )

        wolumeny: list[int] = [
            n.volume
            for n in notowania
        ]

        return float(
            mean(wolumeny)
        )


class ZmiennoscHistoryczna:
    def __call__(
        self,
        historia: HistoriaCen
    ) -> float:

        if len(historia.notowania) < 2:
            return 0.0

        stopy_zwrotu: list[float] = []

        for i in range(
            1,
            len(historia.notowania)
        ):
            poprzednia: float = (
                historia.notowania[i - 1].close
            )

            aktualna: float = (
                historia.notowania[i].close
            )

            if poprzednia == 0:
                continue

            stopa: float = (
                aktualna - poprzednia
            ) / poprzednia

            stopy_zwrotu.append(
                stopa
            )

        if len(stopy_zwrotu) < 2:
            return 0.0

        return (
            pstdev(stopy_zwrotu)
            * 100.0
        )


Kalkulator: type = Callable[
    [HistoriaCen],
    float
]


@dataclass(
    slots=True,
    kw_only=True
)
class KalkulatorPodstawowy:
    historia: HistoriaCen

    srednia_kroczaca: float = field(
        init=False
    )

    stopa_zwrotu: float = field(
        init=False
    )

    sredni_wolumen: float = field(
        init=False
    )

    zmiennosc: float = field(
        init=False
    )

    zakres_cen: ZakresCen = field(
        init=False
    )

    def __post_init__(self) -> None:

        if not self.historia.notowania:
            raise ValueError(
                "historia cen jest pusta"
            )

        kalkulator_sredniej = (
            SredniaKroczaca(okres=20)
        )

        kalkulator_zwrotu = (
            ProstaStopaZwrotu()
        )

        kalkulator_wolumenu = (
            SredniWolumen(okres=20)
        )

        kalkulator_zmiennosci = (
            ZmiennoscHistoryczna()
        )

        self.srednia_kroczaca = (
            kalkulator_sredniej(
                self.historia
            )
        )

        self.stopa_zwrotu = (
            kalkulator_zwrotu(
                self.historia
            )
        )

        self.sredni_wolumen = (
            kalkulator_wolumenu(
                self.historia
            )
        )

        self.zmiennosc = (
            kalkulator_zmiennosci(
                self.historia
            )
        )

        self.zakres_cen = (
            self._oblicz_zakres_cen()
        )

    def _oblicz_zakres_cen(
        self
    ) -> ZakresCen:

        ceny: list[float] = [
            n.close
            for n in self.historia.notowania
        ]

        return ZakresCen(
            minimum=min(ceny),
            maksimum=max(ceny),
            srednia=float(
                mean(ceny)
            )
        )

    def wynik(
        self
    ) -> WynikObliczen:

        return WynikObliczen(
            ticker=self.historia.ticker,
            srednia_kroczaca=(
                self.srednia_kroczaca
            ),
            stopa_zwrotu=(
                self.stopa_zwrotu
            ),
            sredni_wolumen=(
                self.sredni_wolumen
            ),
            zmiennosc=(
                self.zmiennosc
            ),
            liczba_notowan=len(
                self.historia.notowania
            ),
            zakres_cen=(
                self.zakres_cen
            )
        )


def wczytaj_dane_z_modulu_4(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> dict[str, Any]:

    ticker = ticker.strip().upper()

    plik: Path = (
        folder
        / f"{ticker}_typed.json"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 4: {plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8"
    ) as f:

        dane: dict[str, Any] = (
            json.load(f)
        )

    if "spolka" not in dane:
        raise ValueError(
            "brak sekcji spolka"
        )

    if "notowania" not in dane:
        raise ValueError(
            "brak sekcji notowania"
        )

    return dane


def dane_na_historie(
    dane: dict[str, Any]
) -> HistoriaCen:

    spolka_raw: dict[str, Any] = (
        dane["spolka"]
    )

    ticker: str = str(
        dane.get(
            "ticker",
            spolka_raw["ticker"]
        )
    ).strip().upper()

    spolka = Spolka(
        ticker=ticker,
        nazwa=str(
            spolka_raw.get(
                "nazwa",
                ""
            )
        ),
        sektor=str(
            spolka_raw.get(
                "sektor",
                ""
            )
        ),
        branza=(
            None
            if spolka_raw.get("branza") is None
            else str(
                spolka_raw["branza"]
            )
        ),
        kraj=(
            None
            if spolka_raw.get("kraj") is None
            else str(
                spolka_raw["kraj"]
            )
        ),
        tagi=[
            str(x)
            for x in spolka_raw.get(
                "tagi",
                []
            )
        ]
    )

    notowania: list[Notowanie] = []

    for rekord in dane["notowania"]:

        notowanie = Notowanie(
            data=datetime.strptime(
                str(rekord["data"]),
                "%Y-%m-%d"
            ),
            open=float(
                rekord["open"]
            ),
            high=float(
                rekord["high"]
            ),
            low=float(
                rekord["low"]
            ),
            close=float(
                rekord["close"]
            ),
            volume=int(
                rekord["volume"]
            )
        )

        notowania.append(
            notowanie
        )

    return HistoriaCen(
        spolka=spolka,
        notowania=notowania
    )


def wynik_do_dict(
    wynik: WynikObliczen
) -> dict[str, Any]:

    return {
        "ticker":
            wynik.ticker,

        "obliczenia": {
            "srednia_kroczaca_20":
                wynik.srednia_kroczaca,

            "stopa_zwrotu_proc":
                wynik.stopa_zwrotu,

            "sredni_wolumen_20":
                wynik.sredni_wolumen,

            "zmiennosc_proc":
                wynik.zmiennosc,

            "liczba_notowan":
                wynik.liczba_notowan,

            "zakres_cen": {
                "minimum":
                    wynik.zakres_cen.minimum,

                "maksimum":
                    wynik.zakres_cen.maksimum,

                "srednia":
                    wynik.zakres_cen.srednia
            }
        },

        "timestamp":
            wynik.timestamp.isoformat()
    }


def zapisz_obliczenia(
    wynik: WynikObliczen,
    folder: Path = FOLDER_PROJEKTU
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    plik: Path = (
        folder
        / f"{wynik.ticker}_obliczenia.json"
    )

    dane: dict[str, Any] = (
        wynik_do_dict(
            wynik
        )
    )

    with open(
        plik,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2
        )

    return plik


def run() -> None:

    ticker: str = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "zapisanych przez modul 4..."
    )

    dane: dict[str, Any] = (
        wczytaj_dane_z_modulu_4(
            ticker
        )
    )

    historia: HistoriaCen = (
        dane_na_historie(
            dane
        )
    )

    print(
        "utworzono obiekt HistoriaCen"
    )

    print(
        "ticker:",
        historia.ticker
    )

    print(
        "liczba notowan:",
        len(historia.notowania)
    )

    kalkulator = (
        KalkulatorPodstawowy(
            historia=historia
        )
    )

    wynik: WynikObliczen = (
        kalkulator.wynik()
    )

    print(
        "\nSREDNIA KROCZACA 20:",
        round(
            wynik.srednia_kroczaca,
            2
        )
    )

    print(
        "STOPA ZWROTU:",
        round(
            wynik.stopa_zwrotu,
            2
        ),
        "%"
    )

    print(
        "SREDNI WOLUMEN 20:",
        round(
            wynik.sredni_wolumen,
            2
        )
    )

    print(
        "ZMIENNOSC:",
        round(
            wynik.zmiennosc,
            2
        ),
        "%"
    )

    print(
        "MIN CENA:",
        wynik.zakres_cen.minimum
    )

    print(
        "MAX CENA:",
        wynik.zakres_cen.maksimum
    )

    plik: Path = (
        zapisz_obliczenia(
            wynik
        )
    )

    print(
        "\nzapisano dane dla "
        "kolejnego modulu:"
    )

    print(
        plik
    )

    print(
        "\nMODUL OBLICZEN PODSTAWOWYCH "
        "DZIALA POPRAWNIE"
    )